In [ ]:
# 📦 Install required libraries
!pip install rdflib gradio langchain faiss-cpu transformers sentence-transformers

In [ ]:
# 📚 Import libraries
from rdflib import Graph, Literal, RDF, URIRef, Namespace
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.text_splitter import CharacterTextSplitter
from langchain.chains import RetrievalQA
from langchain.llms import HuggingFaceHub
import gradio as gr
import os


In [ ]:
# 🌐 Define RDF Knowledge Graph
EX = Namespace("http://example.org/")
g = Graph()
g.bind("ex", EX)

g.add((EX.Cancer, RDF.type, EX.Disease))
g.add((EX.Cancer, EX.hasTreatment, EX.Chemotherapy))
g.add((EX.Chemotherapy, EX.sideEffect, Literal("Hair Loss")))
g.add((EX.Chemotherapy, EX.sideEffect, Literal("Fatigue")))

def query_kg():
    query = '''
    PREFIX ex: <http://example.org/>
    SELECT ?treatment ?effect WHERE {
        ex:Cancer ex:hasTreatment ?treatment .
        ?treatment ex:sideEffect ?effect .
    }
    '''
    return [(row.treatment.split("/")[-1], str(row.effect)) for row in g.query(query)]


In [ ]:
# 🔍 Load unstructured documents for RAG
def load_docs():
    raw_text = """
    Cancer treatments vary but chemotherapy is a common method. 
    It can lead to several side effects including nausea, fatigue, and hair loss. 
    Other treatments include radiation and immunotherapy depending on the cancer type.
    """
    splitter = CharacterTextSplitter(chunk_size=300, chunk_overlap=30)
    docs = splitter.create_documents([raw_text])
    return docs


In [ ]:
# 🔗 Embed and index documents
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(load_docs(), embeddings)

llm = HuggingFaceHub(
    repo_id="google/flan-t5-large",
    model_kwargs={"temperature": 0.3, "max_length": 256}
)

qa = RetrievalQA.from_chain_type(llm=llm, retriever=vectorstore.as_retriever())


In [ ]:
# 🔀 Hybrid response: combine KG and RAG
def hybrid_answer(question):
    kg_data = query_kg()
    rag_response = qa.run(question)
    facts = "\n".join([f"- {treatment} causes {effect}" for treatment, effect in kg_data])
    return f"📌 Facts from KG:\n{facts}\n\n📖 RAG Answer:\n{rag_response}"


In [ ]:
# 🎨 Gradio Interface
with gr.Blocks() as demo:
    gr.Markdown("## 🧠 Hybrid RAG Chatbot (Knowledge Graph + Unstructured Docs)")
    q_input = gr.Textbox(label="Ask a medical question...")
    output = gr.Textbox(label="Answer")
    btn = gr.Button("Submit")
    btn.click(fn=hybrid_answer, inputs=q_input, outputs=output)

demo.launch()
